# Carvana: one request, from search settings to a pandas row

**Question:** what does one inventory request observe, and how can I inspect and
reload its result? Follow **request preview → saved result → listing rows →
one vehicle's source → price calculation**.

Normal **Restart Kernel and Run All makes no requests and writes no files**.
The default result is the retained September 12, 2026 Chevrolet Tahoe / model year
2023 / ZIP 08542 query: **52 vehicles across three pages**, observed at
14:14:59.794–14:15:05.660 UTC. The optional teaching action observes only the first page,
up to 24 vehicles.

## Two independent selections

| Settings | What changes |
| --- | --- |
| `MAKE`, `MODEL`, `MODEL_YEAR`, `ZIP_CODE`, `LOCATION_FILTER` | The request preview and any separately confirmed teaching request. |
| `SELECTED_REPORT`, `ANALYSIS_CUTOFF` | Which saved result is read, and whether it was available by the cutoff. |
| `EXAMPLE_VIN` | Which vehicle in that saved result is traced; `None` chooses its first row. |
| `COMPARE_BEFORE`, `COMPARE_AFTER` | Optional saved reports for the final comparison section. |

Changing a search setting does not change the saved rows. Check **Selected query**
in the load output to see what was actually collected. Each `rows` record is
one listing observation from one retained page; VIN identifies the vehicle,
`listing_id` identifies Carvana's listing, and `capture_id` links it to that page.

Continue to [20: inventory review](20_carvana_history_analysis.ipynb) after this lab.
The [code walkthrough](../docs/code_walkthrough.md) maps the wider workflow.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if (ROOT / 'vehicle/src').is_dir():
    ROOT = ROOT / 'vehicle'
elif ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'src/vehicle_tracker').is_dir(), 'Open from researchOS, vehicle, or vehicle/notebooks.'
sys.path.insert(0, str(ROOT / 'src'))
from vehicle_tracker.notebook_lab import LabSession, load_capture, load_comparison
from vehicle_tracker.search_evidence import verify_response_evidence

# Ordinary search settings: changing these only changes the preview.
MAKE = 'Chevrolet'
MODEL = 'Tahoe'                         # Native parentModel, not a trim name.
MODEL_YEAR = 2023
ZIP_CODE = '08542'                      # A string preserves the leading zero.
LOCATION_FILTER = False

# Ordinary evidence settings: select an existing saved report to inspect.
SELECTED_REPORT = ROOT / 'data/experiments/mvp_inventory_10k_acceptance_20260912/q101_chevrolet_2023_tahoe/run_report.json'
ANALYSIS_CUTOFF = None                  # None: inspect the selected retained evidence in full.
EXAMPLE_VIN = None                      # None: trace the first admitted row; or enter a VIN.
COMPARE_BEFORE = None                   # Set BOTH to two separate saved run_report.json paths.
COMPARE_AFTER = None

# Advanced test/replay overrides are separate from the ordinary settings above.
SELECTED_REPORT = Path(globals().get('LAB_REPORT_PATH_OVERRIDE', SELECTED_REPORT))
ANALYSIS_CUTOFF = globals().get('AS_OF_OVERRIDE', ANALYSIS_CUTOFF)

# Keep the same allowance and pacing when rerunning cells in this kernel.
if 'LAB_SESSION' not in globals():
    LAB_SESSION = LabSession(ROOT)
print('Selected evidence:', SELECTED_REPORT)
print('Analysis cutoff:', ANALYSIS_CUTOFF or 'All observations in the selected retained report; no new observation.')

## 1. Inspect the exact request before deciding to collect

`filters` selects one make, parent model and model year. `pagination` fixes page 1
and 24 rows per page; `MostPopular` is the observed website ordering, not random
sampling. `zip5` supplies search/delivery context, **not necessarily the vehicle's
physical location**. With location filtering off, the location prefilter feature
is omitted. Neither choice establishes national coverage.

The endpoint is an observed, undocumented public search endpoint. The existing
collector applies its access rules; a failed request does not trigger a fallback.
The preview creates no directory. A live action displays its final destination
and remaining allowance again before asking for fresh confirmation.

In [ ]:
search_settings = dict(make=MAKE, model=MODEL, year=MODEL_YEAR,
                       zip_code=ZIP_CODE, location_filter=LOCATION_FILTER)
request_preview = LAB_SESSION.preview(**search_settings)
print(json.dumps(request_preview, indent=2))

## 2. Load a saved result, without another request

`load_capture` follows the selected report to its saved page files, verifies
their fingerprints, and checks that the read-only SQLite rows agree. The helper
handles those evidence checks; the inspection and price arithmetic remain in
the following pandas cells. Reloading keeps the original observation timestamps.

Read the outputs in this order:

| Output | One row represents | First fields to inspect |
| --- | --- | --- |
| `loaded['summary']` | The selected query attempt | `status`, `query_complete`, reported total, admitted rows and observation interval |
| `loaded['pages']` | One attempted page | HTTP outcome, retained/stored row counts, source path and error |
| `rows` | One admitted listing observation | Retailer, listing ID, VIN, asking price and pending flag |

The historical default is a complete three-page query; a teaching capture normally
contains one sampled page. Completeness applies to the declared query, not national
inventory. A validated zero-result query differs from failure or missing evidence.
A blocked action supplies no rows for calculations and remains visible in its report.


In [ ]:
loaded = load_capture(SELECTED_REPORT, as_of=ANALYSIS_CUTOFF)
rows = loaded['rows']
display(loaded['summary'].T.rename(columns={0: 'selected result'}))
display(loaded['pages'])
print('Selected query:', loaded['report']['filters'], 'ZIP', loaded['report']['zip_code'],
      'location_filter =', loaded['report']['location_filter'])
print('Complete declared query' if loaded['report']['query_complete'] else
      'Incomplete query: only the admitted observed pages are known.')

## 3. Inspect the normalized columns and native states

`head(12)` previews the admitted listing rows; it does not restrict the later
calculation to those twelve. The next tables count missing values and show every
duplicated retailer/VIN or retailer/listing key before a vehicle is selected.

`purchase_pending` keeps the source's purchase-pending flag:
`True` means the flag was set, `False` means it was not, and missing means
unknown. Lock, purchase type and inventory type also retain native source values.
They do not replace separately observed website status or establish a transaction.

The visible calculation is:

`pending_fraction = pending rows / rows with a known pending flag`

Read `known_flags` and `unknown_flags` beside that fraction. Unknown flags
do not enter the denominator as false. This describes the selected result;
it is not a sales estimate. Asking prices remain advertised offers in USD.


In [ ]:
inspect_columns = ['retailer', 'listing_id', 'vin', 'make', 'model', 'year',
                   'mileage_miles', 'asking_price_usd', 'purchase_pending',
                   'vehicle_lock_type', 'purchase_type', 'inventory_type', 'on_demand']
display(rows[inspect_columns].head(12))
display(rows[inspect_columns].isna().sum().rename('missing values').to_frame())
duplicate_vins = rows[rows.duplicated(['retailer', 'vin'], keep=False)]
duplicate_listings = rows[rows.duplicated(['retailer', 'listing_id'], keep=False)]
display(duplicate_vins[inspect_columns], duplicate_listings[inspect_columns])
known_pending = rows['purchase_pending'].notna()
pending_count = rows.loc[known_pending, 'purchase_pending'].eq(True).sum()
pending_denominator = int(known_pending.sum())
pending_fraction = pending_count / pending_denominator if pending_denominator else None
display(pd.DataFrame([dict(pending_rows=int(pending_count), known_flags=pending_denominator,
                          unknown_flags=int((~known_pending).sum()), pending_fraction=pending_fraction)]))

In [ ]:
source_mapping = pd.DataFrame([
    ('inventory.vehicles[].vehicleId', 'listing_id', 'Identifier stored as text'),
    ('inventory.vehicles[].vin', 'vin', 'Vehicle identity within retailer'),
    ('inventory.vehicles[].make / model / year', 'make / model / year', 'Native vehicle description'),
    ('inventory.vehicles[].mileage', 'mileage_miles', 'Odometer miles'),
    ('inventory.vehicles[].price.total', 'asking_price_usd', 'Asking USD; missing remains missing'),
    ('inventory.vehicles[].isPurchasePending', 'purchase_pending', 'Native boolean, never a sale'),
    ('inventory.vehicles[].vehicleLockType', 'vehicle_lock_type', 'Native value, not inferred status'),
    ('inventory.vehicles[].vehiclePurchaseType', 'purchase_type', 'Native purchase-status field'),
    ('inventory.vehicles[].vehicleInventoryType', 'inventory_type', 'Native inventory-type field'),
    ('inventory.vehicles[].isOnDemand', 'on_demand', 'Native boolean'),
    ('collector response-received time', 'observed_at_utc', 'Observation clock, not transaction time'),
], columns=['retained source field', 'analysis column', 'meaning'])
display(source_mapping)

## 4. Trace one real VIN and reproduce a calculation

For the default evidence, the first row is VIN **1GNSCNKD1PR458818**, listing
**4682291**, with a **$40,590 asking price** and pending flag `false`.

Follow the names in the next cells: `trace_row` selects one normalized observation;
its `capture_id` finds `projection_path`; that file points to the retained
source; `source_vehicle` selects the same VIN in that source. The displayed
native record, projection and normalized row all describe the same observation.

The calculation first checks that the source price maps to the normalized number.
Numeric text is parsed for this comparison while the original source value stays
visible. Then:

`price_minus_sample_mean = this asking price - mean of all known asking prices`

`known_price_rows` is that mean's denominator. A negative result means this
vehicle is priced below this selected sample's mean; it does not measure a price
cut, a bargain, or a transaction.

`response_sources/` stores the retained source representation; `raw/` stores
the smaller projection used by the parser. In this historical example,
`source_kind = selected_source`: neither is the full original HTTP body.
For other captures, only `original_response_content` denotes retained complete
`Response.content` bytes under the existing public-field contract.


In [ ]:
source_vehicle = None
trace_row = None
if rows.empty:
    print('No admitted rows to trace. Read the outcome above; do not treat failure as zero inventory.')
else:
    trace_vin = EXAMPLE_VIN if EXAMPLE_VIN is not None else rows['vin'].iloc[0]
    selected = rows.loc[rows['vin'].eq(trace_vin)]
    if len(selected) != 1:
        raise ValueError('Choose a VIN present exactly once in this selected result.')
    trace_row = selected.iloc[0]
    capture_row = loaded['captures'].loc[loaded['captures']['capture_id'].eq(trace_row['capture_id'])].iloc[0]
    projection_path = Path(capture_row['source_path'])
    projection = json.loads(projection_path.read_text(encoding='utf-8'))
    source = verify_response_evidence(projection['response_evidence'])
    source_vehicle = next(v for v in source['inventory']['vehicles'] if v.get('vin') == trace_vin)
    projected_vehicle = next(v for v in projection['vehicles'] if v.get('vin') == trace_vin)
    print('Selected source:', projection['response_evidence']['source_path'])
    print('Projection:', projection_path)
    print('Original observation:', trace_row['observed_at_utc'])
    display(pd.json_normalize([source_vehicle]), pd.json_normalize([projected_vehicle]))
    display(selected[inspect_columns])

In [ ]:
priced_rows = rows.loc[rows['asking_price_usd'].notna(), ['retailer', 'vin', 'asking_price_usd']]
mean_asking_price = priced_rows['asking_price_usd'].mean()  # Only known prices enter this denominator.
if trace_row is not None:
    source_price = source_vehicle.get('price', {}).get('total')
    normalized_price = trace_row['asking_price_usd']
    source_price_numeric = pd.to_numeric(source_price, errors='raise')  # Numeric text uses the parser's rule.
    prices_agree = (pd.isna(source_price_numeric) and pd.isna(normalized_price)) or source_price_numeric == normalized_price
    assert prices_agree, 'Inspect the source-to-row price mapping.'
    price_minus_sample_mean = normalized_price - mean_asking_price
    display(pd.DataFrame([dict(vin=trace_row['vin'], source_asking_usd=source_price,
        normalized_asking_usd=normalized_price, known_price_rows=len(priced_rows),
        selected_result_mean_usd=mean_asking_price, price_minus_mean_usd=price_minus_sample_mean)]))

## 5. Optional: deliberately make one teaching request

Defining `fetch_one_page()` below does nothing live. To collect, deliberately
uncomment and run the example invocation. It displays the exact search, one-request
limit, new isolated destination and remaining teaching budget, then asks you to
type a newly generated `FETCH …` phrase. A stored phrase or a previous confirmation
does not approve another action. Any other input cancels before creating files.

Each action retains the **whole first page**, at most 24 rows. It makes at most one
inventory request, with no pagination or automatic network retries. Three attempted
requests share the existing collector's pacing of at least three seconds between
starts. **Failed or uncertain reserved attempts count too.** Access, identity,
schema and storage failures stop this teaching session; inspect the saved report
and attempt journal instead of trying again. A storage failure before a request
can leave a partial destination; `LAB_SESSION.last_destination` locates it.

Evidence goes only to a new action directory beneath
`vehicle/data/experiments/carvana_notebook_lab/`. It is not imported into the daily
history. These teaching limits are separate from all operating plans and budgets.

The budget is in memory, lasts at most one hour from session creation, and survives
ordinary cell reruns. **Restarting the kernel or manually constructing a new session
loses the counter and pacing history.** It is not a durable limit across kernels or
notebooks. Do not use a reset to retry an access block or replenish an exhausted
allowance; a further teaching session is a separate deliberate decision. Keep one
lab kernel at a time.

In [ ]:
def fetch_one_page():
    # Read the ordinary settings at invocation; confirmation binds this exact snapshot.
    return LAB_SESSION.fetch_one_page(make=MAKE, model=MODEL, year=MODEL_YEAR,
                                     zip_code=ZIP_CODE, location_filter=LOCATION_FILTER)

print('Teaching attempts used:', LAB_SESSION.budget.requests,
      '| remaining:', LAB_SESSION.remaining, '| stopped:', LAB_SESSION.budget.stopped)

# Explicit live action: leave commented during ordinary Run All and offline review.
# live_report_path = fetch_one_page()
# if live_report_path is not None:
#     SELECTED_REPORT = live_report_path
#     display(load_capture(SELECTED_REPORT)['summary'].T)
# Then rerun the lab-load cell and subsequent inspection/calculation cells.
# To reload in a future kernel, put this saved path in SELECTED_REPORT above.
# If incomplete storage prevents a validated reload, inspect the original diagnostic report:
# print((LAB_SESSION.last_destination / 'run_report.json').read_text(encoding='utf-8'))
# If no report could be published, keep the partial directory and inspect the raised error.

## 6. Optional: compare two separately collected results of the same query

Set both report paths, earlier first, then rerun this section. The reader requires
the same query filters, ZIP, location setting and sort, with later non-overlapping
observation times. It rejects a blocked capture or the same observation twice.

The outer join has **one row per retailer/VIN across the two saved results**.
It retains `listing_id_before` and `listing_id_after` so a relisting remains
visible. `both` means observed in both results; `left_only` and
`right_only` mean observed in only one. Price subtraction uses shared VINs.

A VIN absent from a sampled page may have moved to another page or ordering
position. Before-only and after-only membership therefore does not establish a
full-inventory exit or entry. Missing prices stay missing; none of these results
establishes a sale.


In [ ]:
if COMPARE_BEFORE is None and COMPARE_AFTER is None:
    print('Comparison not selected. Choose two separately collected reports from the same query.')
elif COMPARE_BEFORE is None or COMPARE_AFTER is None:
    raise ValueError('Select both comparison report paths.')
else:
    before, after = load_comparison(COMPARE_BEFORE, COMPARE_AFTER, as_of=ANALYSIS_CUTOFF)
    intervals = pd.concat([before['summary'].assign(selection='before'),
                           after['summary'].assign(selection='after')], ignore_index=True)
    display(intervals[['selection', 'observation_start', 'observation_end', 'query_complete',
                       'retained_pages', 'admitted_rows', 'reported_query_total']])
    seconds_between_observations = (pd.Timestamp(after['run']['observation_start']) -
                                   pd.Timestamp(before['run']['observation_end'])).total_seconds()
    print('Seconds from earlier interval end to later interval start:', seconds_between_observations)
    compare_columns = ['retailer', 'vin', 'listing_id', 'asking_price_usd', 'observed_at_utc']
    paired = before['rows'][compare_columns].merge(after['rows'][compare_columns],
        on=['retailer', 'vin'], how='outer', suffixes=('_before', '_after'),
        indicator=True, validate='one_to_one')
    display(paired['_merge'].value_counts().rename(index={'left_only': 'before-only observed',
        'right_only': 'after-only observed', 'both': 'shared observed VINs'}).to_frame('VINs'))
    shared_vins = paired.loc[paired['_merge'].eq('both')].copy()
    shared_vins['asking_price_change_usd'] = (shared_vins['asking_price_usd_after'] -
                                             shared_vins['asking_price_usd_before'])
    shared_vins['listing_id_changed'] = shared_vins['listing_id_before'].ne(shared_vins['listing_id_after'])
    display(shared_vins.drop(columns='_merge'))

## Three short exercises

1. **Predict a request change, offline.** Change `MODEL_YEAR` to 2022 and rerun only
   the request preview (refresh `search_settings` in that cell). Expect both year
   bounds to change, with page 1, 24 rows and ZIP unchanged. No new evidence is
   collected, and the saved result still describes its original query.
2. **Inspect one explicit request.** When you deliberately choose to collect, run
   `fetch_one_page()` and confirm its fresh phrase. Inspect its HTTP outcome and
   page information, then trace a VIN's native pending/purchase fields. Expect up
   to 24 rows and often an incomplete query. This supports a dated inventory-page
   observation, not a confirmed sale or a national count. Stop on failure.
3. **Reproduce the calculation offline.** Save the report path in `SELECTED_REPORT`,
   leave live invocations commented, and restart/run all. Verify the observation
   timestamp and the VIN's asking price minus the mean of known prices. Expect
   the same result from the same evidence; reloading makes no fresh observation.

For routine operation continue to **20 for inventory/pricing** and **24 for the
selected status experiment**. Review **30** for saved forecasts and quarterly
assumptions. Live operating collection and the seven-date validation remain
separate authorized actions; this lab neither launches nor schedules them.